# 00 · Live Pull

Pulls today's OHLCV from **yfinance** for the full ticker universe and writes a
Parquet file tagged `_source_type='live'` to `/Volumes/asset_mgmt/bronze/uploads/`.

Runs are a no-op on weekends and market holidays (empty yfinance response).

---

### Databricks job schedule

In the **Jobs** UI → *Edit schedule* → set:

| Field | Value |
|---|---|
| Schedule type | Cron |
| Cron expression | `30 22 * * *` |
| Timezone | **Europe/Paris** (covers CET/CEST automatically) |

At 22:30 CET the NYSE/NASDAQ close (16:00 ET) has already settled, so yfinance
returns the full daily bar.


In [ ]:
%pip install yfinance --quiet

In [ ]:
from datetime import date, timedelta
import pandas as pd
import yfinance as yf
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, LongType
)

TICKERS = [
    # Tech
    "AAPL", "MSFT", "NVDA", "GOOGL", "META",
    # Finance
    "JPM", "BAC", "GS", "MS", "WFC",
    # Healthcare
    "JNJ", "UNH", "PFE", "ABBV", "MRK",
    # Energy
    "XOM", "CVX", "COP", "SLB", "EOG",
    # Consumer
    "AMZN", "HD", "MCD", "NKE", "COST",
    # Benchmark
    "SPY",
]

UPLOADS_PATH = "/Volumes/asset_mgmt/bronze/uploads/"
TODAY        = date.today()
TOMORROW     = TODAY + timedelta(days=1)

print(f"Run date : {TODAY}  ({TODAY.strftime('%A')})")
print(f"Tickers  : {len(TICKERS)}")

In [ ]:
# ── Weekend guard ─────────────────────────────────────────────────────────
# weekday() returns 5=Saturday, 6=Sunday
if TODAY.weekday() >= 5:
    day_name = TODAY.strftime("%A")
    print(f"[SKIP] Today is {day_name} — US markets are closed. Nothing to pull.")
    dbutils.notebook.exit(f"skipped:{day_name}")

In [ ]:
# ── Pull each ticker individually so one failure doesn't block the rest ───
frames = []
failed = []

for ticker in TICKERS:
    try:
        df = yf.download(
            ticker,
            start=TODAY.isoformat(),
            end=TOMORROW.isoformat(),
            auto_adjust=True,
            progress=False,
        )

        if df.empty:
            # Holiday or data not yet available — not an error
            print(f"  [SKIP] {ticker}: no data (holiday or data lag)")
            continue

        df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
        df.index.name = "Date"
        df.reset_index(inplace=True)
        df.insert(0, "Ticker", ticker)
        frames.append(df)

    except Exception as exc:
        print(f"  [ERROR] {ticker}: {exc}")
        failed.append(ticker)

print(f"\nPulled   : {len(frames)} tickers with data")
print(f"Skipped  : {len(TICKERS) - len(frames) - len(failed)} (holiday/no data)")
print(f"Failed   : {len(failed)} — {failed if failed else 'none'}")

In [ ]:
# ── Holiday guard: exit cleanly if every ticker came back empty ───────────
if not frames:
    print(f"[SKIP] No data returned for any ticker on {TODAY} — likely a market holiday.")
    dbutils.notebook.exit(f"skipped:holiday:{TODAY}")

In [ ]:
# ── Assemble pandas DataFrame and convert to Spark ────────────────────────
pandas_df = pd.concat(frames, ignore_index=True)
pandas_df["Date"] = pd.to_datetime(pandas_df["Date"]).dt.date

schema = StructType([
    StructField("Ticker", StringType(),  False),
    StructField("Date",   DateType(),    False),
    StructField("Open",   DoubleType(),  True),
    StructField("High",   DoubleType(),  True),
    StructField("Low",    DoubleType(),  True),
    StructField("Close",  DoubleType(),  True),
    StructField("Volume", LongType(),    True),
])

spark_df = (
    spark.createDataFrame(pandas_df, schema=schema)
        .withColumn("_source_type",      F.lit("live"))
        .withColumn("_ingest_timestamp", F.current_timestamp())
)

print(f"Rows assembled : {spark_df.count()}")
display(spark_df)

In [ ]:
# ── Write Parquet to uploads volume ───────────────────────────────────────
# One file per day: live_YYYY-MM-DD.parquet
# coalesce(1) produces a single file; the job runs once per day so size is small.
out_path = f"{UPLOADS_PATH}live_{TODAY}.parquet"

(
    spark_df
        .coalesce(1)
        .write
        .mode("overwrite")   # idempotent: re-running the job on the same day is safe
        .parquet(out_path)
)

print(f"Written → {out_path}")
dbutils.notebook.exit(f"ok:{TODAY}:{spark_df.count()} rows")